# Data Fusion 2025 — Задача 2 "4cast"
## Прогнозирование еженедельных переводов юридических лиц

**Задача:** предсказать суммарные переводы 51 963 клиентов банка за 12 недель (118–129),  
используя историю за 118 недель (0–117).

**Метрика:** средний по клиентам RMSLE:
$$\overline{RMSLE} = \frac{1}{N}\sum_{i=1}^N \sqrt{\frac{1}{T}\sum_{t=1}^T (\log(1+y_{it}) - \log(1+\hat{y}_{it}))^2}$$

**Public leaderboard:** недели 118–121 (4 из 12 прогнозных).

---


## Скачивание данных

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import lightgbm as lgb
import warnings
import json
from pathlib import Path
import gc, time

warnings.filterwarnings("ignore")
plt.rcParams["figure.figsize"] = (14, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.4
sns.set_palette("tab10")

DATA_DIR = Path("data")
FIG_DIR  = Path("figures1")
FIG_DIR.mkdir(exist_ok=True)

# Splits (из calendar_extended.csv)
TRAIN_WEEKS    = list(range(0, 106))     # 0–105  → train
VAL_PUB_WEEKS  = list(range(106, 110))  # 106–109 → validation_public
VAL_PRIV_WEEKS = list(range(110, 118))  # 110–117 → validation_private
PUBLIC_WEEKS   = list(range(118, 122))  # 118–121 → public leaderboard
PRIVATE_WEEKS  = list(range(122, 130))  # 122–129 → private leaderboard
FORECAST_WEEKS = list(range(118, 130))  # все 12 недель прогноза

print("Config loaded.")


Config loaded.


In [3]:
print("Loading data...")
ts_train = pd.read_parquet(DATA_DIR / "target_series_extended.parquet")       # weeks 0–117
calendar = pd.read_csv(DATA_DIR / "calendar_extended.csv", parse_dates=["date"])
sample   = pd.read_csv(DATA_DIR / "sample_submit_extended.csv")

ts_train["week"] = ts_train["week"].astype(int)

TX_FILES = sorted(DATA_DIR.glob("transactions_[1-5].parquet"))
print(f"Found {len(TX_FILES)} transaction files: {[f.name for f in TX_FILES]}")

t0 = time.time()
trns_chunks = []
for f in TX_FILES:
    print(f"Reading {f.name} ...")
    chunk = pd.read_parquet(f)
    print(f"  -> {len(chunk):,} rows")
    trns_chunks.append(chunk)

trns_raw = pd.concat(trns_chunks, ignore_index=True)
del trns_chunks
gc.collect()

Loading data...
Found 5 transaction files: ['transactions_1.parquet', 'transactions_2.parquet', 'transactions_3.parquet', 'transactions_4.parquet', 'transactions_5.parquet']
Reading transactions_1.parquet ...
  -> 52,089,892 rows
Reading transactions_2.parquet ...
  -> 55,045,806 rows
Reading transactions_3.parquet ...
  -> 59,510,350 rows
Reading transactions_4.parquet ...
  -> 61,492,247 rows
Reading transactions_5.parquet ...
  -> 30,036,951 rows


0

## Подготовка данных 

Разбиваем датасет с транзакциями на 117 файлов, по неделе на файл для оптимизации. 

In [4]:
trns = trns_raw.copy()

In [ ]:
#trns = trns.iloc[:50000]

In [5]:
trns["date"] = pd.to_datetime(trns["date"], utc=True).dt.tz_localize(None).dt.normalize()

all_inn = ts_train["inn_id"].unique()

cal_117 = calendar.loc[calendar["week"] <= 117].copy()
cal_117["date"] = pd.to_datetime(cal_117["date"]).dt.tz_localize(None).dt.normalize()
all_dates = np.sort(cal_117["date"].drop_duplicates().values)

mi = pd.MultiIndex.from_product([all_inn, all_dates], names=["doc_payer_inn", "date"])
new_trns = mi.to_frame(index=False)

new_trns["doc_payee_inn"] = ""
new_trns["trns_count"] = 1.0
new_trns["trns_amount"] = 0
new_trns["doc_payer_bank_name_encoded"] = 51
new_trns["doc_payee_bank_name_encoded"] = 0
new_trns["doc_payer_bank_name_flag"] = 1
new_trns["doc_payee_bank_name_flag"] = 0
new_trns["trns_class_encoded"] = 0

trns = pd.concat([trns, new_trns], ignore_index=True)

trns = pd.merge(trns, calendar, on='date', how='left')

allowed = ts_train["inn_id"].unique()
trns = trns[trns["doc_payer_inn"].isin(allowed)]

In [6]:
trns = trns.groupby(["week", "doc_payer_inn"], as_index=False, sort=False).agg({
    "trns_amount": "sum",
    "part": "first",
}).rename(columns={
                 "doc_payer_inn": "inn_id",
                 "trns_amount": "target",
             })

In [7]:
trns.head()

,week,inn_id,target,part
0,0,inn1000051,4.221593e+05,train
1,0,inn1000367,3.994532e+05,train
2,0,inn1000484,9.672847e+05,train
3,0,inn1000624,1.684209e+05,train
4,0,inn1000884,5.724560e+06,train


In [ ]:
expected_weeks = sorted(trns["week"].astype(int).unique().tolist())

expected_len = trns["inn_id"].nunique()

dup_mask = trns.duplicated(subset=["inn_id", "week"], keep=False)
if dup_mask.any():
    dup_count = int(dup_mask.sum())
    raise ValueError(f"Найдены дубли по (inn_id, week): {dup_count} строк")

min_week, max_week = expected_weeks[0], expected_weeks[-1]
full_range = list(range(min_week, max_week + 1))
if expected_weeks != full_range:
    missing_weeks = sorted(set(full_range) - set(expected_weeks))
    raise ValueError(f"Неполный диапазон недель. Отсутствуют: {missing_weeks}")

out_dir = Path("transactions_chunks")
out_dir.mkdir(parents=True, exist_ok=True)

trns_sorted = trns.sort_values(["week", "inn_id"]).reset_index(drop=True)
errors = []
written_files = 0

for week, chunk in trns_sorted.groupby("week", sort=True):
    chunk_len = len(chunk)
    if chunk_len != expected_len:
        errors.append(f"week={int(week)}: len={chunk_len}, expected={expected_len}")
        continue

    out_path = out_dir / f"week_{int(week):03d}.parquet"
    chunk.to_parquet(out_path, index=False)
    written_files += 1

if errors:
    preview = "\n".join(errors[:10])
    raise ValueError(f"Обнаружены недели с неверной длиной чанка:\n{preview}")

print(f"Done: записано {written_files} файлов в {out_dir.resolve()}")

Done: записано 118 файлов в /Users/aeseverenkova/Desktop/bank_forecasting/transactions_chunks


## Построение признаков

Хотим в признаках зафиксировать связь предыдущих событий на будующие. Для этого будем использовать лаги.

In [113]:
def _lags_row_at_cutoff(n: int, weeks: np.ndarray, y: np.ndarray, max_lag: int = 70):
    """Одна строка лагов для недели n (эквивалент shift)"""

    mask = (weeks <= n) & (weeks >= n - max_lag)
    idx = np.flatnonzero(mask)

    w = weeks[idx]
    yy = y[idx]
    pos = int(np.where(w == n)[0][0])

    row = {}
    for lag in range(max_lag):
        j = pos - lag
        row[f"lag_{lag}_week_{n}"] = float(yy[j]) if j >= 0 else 0.0

    return row

def get_features(n, df) -> pd.DataFrame:
    """
    n - номер недели
    df - датафрейм с транзакциями одного inn_id
    """

    max_lag = 70

    g = df.sort_values("week")
    weeks = g["week"].to_numpy()
    y = g["target"].to_numpy(dtype=float)
    row = _lags_row_at_cutoff(n, weeks, y, max_lag)
    
    return pd.DataFrame([row]) if row else pd.DataFrame()


In [128]:
def build_features_horizons_for_inn(
    n_target: int, k: int, inn_df: pd.DataFrame, max_lag: int = 70
) -> pd.DataFrame:
    """Все горизонты i=1..k для одного ИНН """

    g = inn_df.sort_values("week")
    weeks = g["week"].to_numpy()
    y = g["target"].to_numpy(dtype=float)

    rows = []
    for i in range(1, k + 1):
        cutoff = n_target - i
        row = _lags_row_at_cutoff(cutoff, weeks, y, max_lag)
        rows.append(row)
        
    return pd.DataFrame(rows) if rows else pd.DataFrame()

Представим, что мы хотим предсказать на одну неделю вперед. У нас есть данные 0-117 недель. Тогда мы будем считать 117 неделю таргетом, а строить признаки на 0-116 неделях. 

Если мы хотим предсказать 2 неделю, таргетом снова считаем 117 неделю, но признаки уже строим на 0-115 неделях. 

Если мы хотим предсказать 3 неделю, мы строрим признаки на 0-114 неделях. 

Тогда чтобы предсказать неделю k надо строить признаки на неделях 117-k. Так мы будем показывать как влияют предыдущие значения на будующую неделю номер k. 

Предсказание на 12 недель: предсказание на 1, предсказание на 2, .... предсказание на 12 неделю.

In [129]:
train = pd.DataFrame()
n = 117 # текущая неделя 
k = 12 # горизонт предсказания

train_list = []

df_week_n = trns.loc[trns["week"] == n, ["inn_id", "target"]]
inn_subset = df_week_n["inn_id"].unique()
inn_groups = {gid: g for gid, g in trns.groupby("inn_id", sort=False)}

train_list = []
for inn in inn_subset:
    inn_df = inn_groups.get(inn)

    t_series = df_week_n.loc[df_week_n["inn_id"] == inn, "target"]
    t_val = t_series.iloc[0]

    f_block = build_features_horizons_for_inn(n, k, inn_df)
    f_block["target"] = t_val
    train_list.append(f_block)
        
train = pd.concat(train_list, ignore_index=True) if train_list else pd.DataFrame()

train.head()

,lag_0_week_116,lag_1_week_116,lag_2_week_116,lag_3_week_116,lag_4_week_116,lag_5_week_116,lag_6_week_116,lag_7_week_116,lag_8_week_116,lag_9_week_116,...,lag_61_week_105,lag_62_week_105,lag_63_week_105,lag_64_week_105,lag_65_week_105,lag_66_week_105,lag_67_week_105,lag_68_week_105,lag_69_week_105,target
0,136050.637695,310471.976562,677862.664062,260502.425293,31496.517883,39889.730469,5.958537e+06,2.508962e+06,21638.005859,57638.035156,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,72807.546143
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,72807.546143
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,72807.546143
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,72807.546143
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,72807.546143


Чтобы увеличить обучающую выборку, попробуем не фиксировать таргетную неделю на n = 117. Построим по схеме выше признаки для 20 таргенных недель. 

In [130]:
def sample_n(n, df) -> pd.DataFrame:
    k = 12 # горизонт предсказания

    df_week_n = df.loc[df["week"] == n, ["inn_id", "target"]]
    inn_groups = {gid: g for gid, g in df.groupby("inn_id", sort=False)}

    train_list = []
    for inn in df_week_n["inn_id"].unique():
        inn_df = inn_groups.get(inn)

        t_series = df_week_n.loc[df_week_n["inn_id"] == inn, "target"]
        t_val = t_series.iloc[0]

        f_block = build_features_horizons_for_inn(n, k, inn_df)
        f_block["target"] = t_val
        train_list.append(f_block)

    return pd.concat(train_list, ignore_index=True) if train_list else pd.DataFrame()

In [ ]:
N = 20
train_parts = []

for i in range(N):
    n = 117 - i
    train_parts.append(sample_n(n, trns))

train = pd.concat(train_parts, ignore_index=True) if train_parts else pd.DataFrame()
train.head()

## Обучение

In [137]:
!pip3 install xgboost

Defaulting to user installation because normal site-packages is not writeable
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 2.3 MB/s eta 0:00:00a 0:00:010m

[notice] A new release of pip available: 22.3.1 -> 26.1.1
[notice] To update, run: pip3 install --upgrade pip


In [138]:
import pandas as pd
import xgboost as xgb
from sklearn.metrics import mean_absolute_error

In [ ]:
train = train.dropna()

X = train[trns.columns.to_list().remove('target')]
y = train['target']

split = int(len(train) * 0.8)
X_train, X_test = X.iloc[:split], X.iloc[split:]
y_train, y_test = y.iloc[:split], y.iloc[split:]

model = xgb.XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42,
)

model.fit(X_train, y_train)
pred = model.predict(X_test)

KeyError: None

In [ ]:
FEATURE_COLS = [
    "lag_1", "lag_7", "lag_14", "lag_28", "roll_mean_7", "dow", "month", "weekend"
]


def fit_and_predict(train_frame: pd.DataFrame, valid_frame: pd.DataFrame):
    X_train = train_frame[FEATURE_COLS]
    y_train = train_frame["target_daily"]
    X_valid = valid_frame[FEATURE_COLS]
    y_valid = valid_frame["target_daily"]

    model = CatBoostRegressor(
        loss_function="RMSE",
        iterations=1000,
        depth=6,
        learning_rate=0.05,
        random_seed=42,
        verbose=False,
    )
    model.fit(X_train, np.log1p(y_train))

    pred_log = model.predict(X_valid)
    pred = np.expm1(pred_log)
    pred = np.clip(pred, 0, None)
    return y_valid, pred


def group_rmsle(frame: pd.DataFrame) -> float:
    by_client = frame.groupby("inn_id", sort=False)
    client_scores = by_client.apply(
        lambda g: np.sqrt(metrics.mean_squared_log_error(g["y_true"], g["y_pred"]))
    )
    return float(client_scores.mean())

# без leakage
fixed_df = df_daily.sort_values(["inn_id", "date"]).copy()
grouped = fixed_df.groupby("inn_id", sort=False)

fixed_df["lag_1"] = grouped["target_daily"].shift(1)
fixed_df["lag_7"] = grouped["target_daily"].shift(7)
fixed_df["lag_14"] = grouped["target_daily"].shift(14)
fixed_df["lag_28"] = grouped["target_daily"].shift(28)
fixed_df["roll_mean_7"] = grouped["target_daily"].transform(
    lambda s: s.shift(1).rolling(7, min_periods=7).mean()
)
fixed_df["dow"] = fixed_df["date"].dt.dayofweek
fixed_df["month"] = fixed_df["date"].dt.month
fixed_df["weekend"] = fixed_df["dow"].isin([5, 6])

fixed_data = fixed_df.dropna(subset=FEATURE_COLS + ["target_daily"]).copy()
train_mask = fixed_data["week"] <= 109
valid_mask = fixed_data["week"].between(110, 117)

train_df = fixed_data.loc[train_mask].copy()
valid_df = fixed_data.loc[valid_mask].copy()

y_valid, pred = fit_and_predict(train_df, valid_df)

global_rmsle_fixed = float(np.sqrt(metrics.mean_squared_log_error(y_valid, pred)))

pred_frame = valid_df[["inn_id", "date"]].copy()
pred_frame["y_true"] = y_valid.values
pred_frame["y_pred"] = pred
group_rmsle_fixed = group_rmsle(pred_frame)

print(f"A: {global_rmsle_fixed}")
print(f"RMSLE: {group_rmsle_fixed}")

A: 4.509756162583145
RMSLE: 4.407655346201267


In [ ]:
pred

array([1.18132077e+02, 9.72280843e+01, 2.26658308e+03, ...,
       1.81658876e+06, 1.80509779e+02, 1.57568572e+01], shape=(2909928,))